# MTA Service Reliability Analysis
## 04: Hourly Ridership Analysis

---
### Imports

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import PercentFormatter
import matplotlib.dates as mdates

import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import statsmodels

import calplot
import seaborn as sns

---
### Load Data

This takes a little less than a minute because the dataset is large.

In [3]:
hourly = pd.read_csv(
    "../data/raw/hourly_ridership_select_months.csv"
)

hourly.columns.tolist()

['transit_timestamp',
 'transit_mode',
 'station_complex_id',
 'station_complex',
 'borough',
 'payment_method',
 'fare_class_category',
 'ridership',
 'transfers',
 'latitude',
 'longitude',
 'georeference']

In [4]:
hourly.head()

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,georeference
0,2026-04-01T00:00:00.000,subway,438,"135 St (2,3)",Manhattan,omny,OMNY - Fair Fare,4.0,0.0,40.814228,-73.94077,POINT (-73.94077 40.814228)
1,2026-04-01T00:00:00.000,subway,268,"65 St (M,R)",Queens,omny,OMNY - Fair Fare,1.0,0.0,40.749670,-73.89845,POINT (-73.89845 40.74967)
2,2026-04-01T00:00:00.000,subway,164,"34 St-Penn Station (A,C,E)",Manhattan,omny,OMNY - Fair Fare,29.0,1.0,40.752290,-73.99339,POINT (-73.99339 40.75229)
3,2026-04-01T00:00:00.000,subway,192,Rockaway Blvd (A),Queens,omny,OMNY - Other,1.0,0.0,40.680428,-73.84386,POINT (-73.84386 40.680428)
4,2026-04-01T00:00:00.000,subway,213,"Fordham Rd (B,D)",Bronx,metrocard,Metrocard - Unlimited 30-Day,1.0,0.0,40.861298,-73.89775,POINT (-73.89775 40.861298)


In [5]:
hourly.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 5153761 entries, 0 to 5153760
Data columns (total 12 columns):
 #   Column               Dtype  
---  ------               -----  
 0   transit_timestamp    str    
 1   transit_mode         str    
 2   station_complex_id   str    
 3   station_complex      str    
 4   borough              str    
 5   payment_method       str    
 6   fare_class_category  str    
 7   ridership            float64
 8   transfers            float64
 9   latitude             float64
 10  longitude            float64
 11  georeference         str    
dtypes: float64(4), str(8)
memory usage: 2.5 GB


In [6]:
hourly.shape   # 5,153,761 rows!

(5153761, 12)

In [7]:
hourly.describe(include="all")

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,georeference
count,5153761,5153761,5153761,5153761,5153761,5153761,5153761,5.153761e+06,5.153761e+06,5.153761e+06,5.153761e+06,5153761
unique,2184,3,428,428,5,2,12,NaN,NaN,NaN,NaN,428
top,2026-04-14T08:00:00.000,subway,611,"Times Sq-42 St/Port Authority Bus Terminal (1,...",Brooklyn,omny,OMNY - Full Fare,NaN,NaN,NaN,NaN,POINT (-73.98758 40.755745)
freq,3088,5118553,18106,18106,1813783,4086956,913096,NaN,NaN,NaN,NaN,18106
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.697361e+01,3.562264e+00,4.073321e+01,-7.393473e+01,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.097320e+02,2.387731e+01,7.846069e-02,5.623693e-02,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,-5.000000e+00,4.057613e+01,-7.407484e+01,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000e+00,0.000000e+00,4.067834e+01,-7.398123e+01,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.000000e+00,0.000000e+00,4.072561e+01,-7.394550e+01,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.800000e+01,1.000000e+00,4.079502e+01,-7.389865e+01,NaN


---
### Data Cleaning and Feature Engineering

Count missing values.

In [8]:
hourly.isna().sum()

transit_timestamp      0
transit_mode           0
station_complex_id     0
station_complex        0
borough                0
payment_method         0
fare_class_category    0
ridership              0
transfers              0
latitude               0
longitude              0
georeference           0
dtype: int64

In [9]:
# See columns with only missing values.
hourly.isnull().sum()[hourly.isnull().sum() > 0]

Series([], dtype: int64)

None of the values are missing, which is great.

Check if any rows are duplicated.

In [10]:
hourly.duplicated().sum()

np.int64(0)

None of the values are duplicated.

In [11]:
hourly.nunique()   # Unique values in each column

transit_timestamp      2184
transit_mode              3
station_complex_id      428
station_complex         428
borough                   5
payment_method            2
fare_class_category      12
ridership              7362
transfers              1279
latitude                427
longitude               425
georeference            428
dtype: int64

Create new columns from the timestamp for easier analysis later.

In [12]:
hourly["transit_timestamp"] = pd.to_datetime(hourly["transit_timestamp"])

hourly["date"] = hourly["transit_timestamp"].dt.date
hourly["hour"] = hourly["transit_timestamp"].dt.hour
hourly["weekday"] = hourly["transit_timestamp"].dt.day_name()
hourly["month"] = hourly["transit_timestamp"].dt.month_name()
hourly["year"] = hourly["transit_timestamp"].dt.year
hourly["day"] = hourly["transit_timestamp"].dt.day
hourly["week"] = hourly["transit_timestamp"].dt.isocalendar().week

hourly.head()

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers,latitude,longitude,georeference,date,hour,weekday,month,year,day,week
0,2026-04-01,subway,438,"135 St (2,3)",Manhattan,omny,OMNY - Fair Fare,4.0,0.0,40.814228,-73.94077,POINT (-73.94077 40.814228),2026-04-01,0,Wednesday,April,2026,1,14
1,2026-04-01,subway,268,"65 St (M,R)",Queens,omny,OMNY - Fair Fare,1.0,0.0,40.749670,-73.89845,POINT (-73.89845 40.74967),2026-04-01,0,Wednesday,April,2026,1,14
2,2026-04-01,subway,164,"34 St-Penn Station (A,C,E)",Manhattan,omny,OMNY - Fair Fare,29.0,1.0,40.752290,-73.99339,POINT (-73.99339 40.75229),2026-04-01,0,Wednesday,April,2026,1,14
3,2026-04-01,subway,192,Rockaway Blvd (A),Queens,omny,OMNY - Other,1.0,0.0,40.680428,-73.84386,POINT (-73.84386 40.680428),2026-04-01,0,Wednesday,April,2026,1,14
4,2026-04-01,subway,213,"Fordham Rd (B,D)",Bronx,metrocard,Metrocard - Unlimited 30-Day,1.0,0.0,40.861298,-73.89775,POINT (-73.89775 40.861298),2026-04-01,0,Wednesday,April,2026,1,14


---
### Exploratory Data Analysis

In [13]:
hourly["borough"].unique()

<StringArray>
['Manhattan', 'Queens', 'Bronx', 'Brooklyn', 'Staten Island']
Length: 5, dtype: str

In [14]:
hourly["payment_method"].value_counts()

payment_method
omny         4086956
metrocard    1066805
Name: count, dtype: int64

In [15]:
hourly["fare_class_category"].value_counts()

fare_class_category
OMNY - Full Fare                    913096
OMNY - Fair Fare                    813093
OMNY - Students                     797158
OMNY - Seniors & Disability         793316
OMNY - Other                        770293
Metrocard - Other                   627437
Metrocard - Unlimited 30-Day        256162
Metrocard - Unlimited 7-Day          81374
Metrocard - Seniors & Disability     54345
Metrocard - Full Fare                47390
Metrocard - Fair Fare                   83
Metrocard - Students                    14
Name: count, dtype: int64

In [16]:
hourly["transit_mode"].value_counts()

transit_mode
subway                   5118553
staten_island_railway      19630
tram                       15578
Name: count, dtype: int64

Top 3 Largest Stations (by number of observations):

* Times Sq - 42 St
* 34 St-Herald Sq
* Flushing-Main St

In [17]:
hourly["station_complex"].value_counts().head(20)

station_complex
Times Sq-42 St/Port Authority Bus Terminal (1,2,3,7,A,C,E,N,Q,R,W,S)    18106
34 St-Herald Sq (B,D,F,M,N,Q,R,W)                                       16876
Flushing-Main St (7)                                                    16537
Grand Central-42 St (4,5,6,7,S)                                         16504
14 St-Union Sq (4,5,6,L,N,Q,R,W)                                        16455
Jamaica Center-Parsons/Archer (E,J,Z)                                   16148
Fulton St (2,3,4,5,A,C,J,Z)                                             16143
34 St-Penn Station (1,2,3)                                              16061
Sutphin Blvd-Archer Av-JFK Airport (E,J,Z)                              15978
Jackson Hts-Roosevelt Av/74 St-Broadway (7,E,F,M,R)                     15976
34 St-Penn Station (A,C,E)                                              15962
161 St-Yankee Stadium (4,B,D)                                           15631
125 St (4,5,6)                                  

In [18]:
hourly["borough"].value_counts()

borough
Brooklyn         1813783
Manhattan        1575941
Queens            913314
Bronx             831093
Staten Island      19630
Name: count, dtype: int64

The most hourly observations happened in Brooklyn and Manhattan, suggesting there are more subway lines there.

Due to the large size, the dataset covers April, May, June 2026 only.

In [19]:
hourly["transit_timestamp"].min()

Timestamp('2026-04-01 00:00:00')

In [20]:
hourly["transit_timestamp"].max()

Timestamp('2026-06-30 23:00:00')

In [21]:
hourly["ridership"].describe()

count    5.153761e+06
mean     6.697361e+01
std      3.097320e+02
min      1.000000e+00
25%      2.000000e+00
50%      8.000000e+00
75%      2.800000e+01
max      2.187800e+04
Name: ridership, dtype: float64

In [22]:
hourly["transfers"].describe()

count    5.153761e+06
mean     3.562264e+00
std      2.387731e+01
min     -5.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      2.556000e+03
Name: transfers, dtype: float64

In [23]:
hourly[["ridership","transfers"]].corr()

,ridership,transfers
ridership,1.000000,0.368338
transfers,0.368338,1.000000


Top 3 Largest Stations (by ridership numbers):

* Times Sq - 42 St
* Gran Central - 42 St
* 34 St-Herald Sq

In [24]:
(
    hourly
    .groupby("station_complex")["ridership"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

station_complex
Times Sq-42 St/Port Authority Bus Terminal (1,2,3,7,A,C,E,N,Q,R,W,S)    12658531.0
Grand Central-42 St (4,5,6,7,S)                                          9646217.0
34 St-Herald Sq (B,D,F,M,N,Q,R,W)                                        6986562.0
14 St-Union Sq (4,5,6,L,N,Q,R,W)                                         6483943.0
34 St-Penn Station (A,C,E)                                               5603355.0
Fulton St (2,3,4,5,A,C,J,Z)                                              5154223.0
34 St-Penn Station (1,2,3)                                               4745884.0
59 St-Columbus Circle (1,A,C,B,D)                                        4559982.0
Jackson Hts-Roosevelt Av/74 St-Broadway (7,E,F,M,R)                      4002444.0
Flushing-Main St (7)                                                     3926173.0
Lexington Av/51-53 Sts (6,E,F)                                           3760319.0
Chambers St/WTC/Park Place/Cortlandt St (2,3,A,C,E,R,W)                

Interestingly enough, there was more ridership in Brooklyn than Manhattan. It makes sense because the top 3 largest stations by ridership numbers are located in Manhattan.

In [25]:
(
    hourly
    .groupby("borough")["ridership"]
    .sum()
    .sort_values(ascending=False)
)

borough
Manhattan        193460663.0
Brooklyn          77359305.0
Queens            50446037.0
Bronx             23312663.0
Staten Island       587331.0
Name: ridership, dtype: float64

The peak hours are 7-9 AM and 2-6 PM.

In [26]:
(
    hourly
    .groupby("hour")["ridership"]
    .sum()
)

hour
0      3387755.0
1      1481858.0
2       890442.0
3       833052.0
4      1740973.0
5      5284684.0
6     11426867.0
7     21336351.0
8     27423309.0
9     19414026.0
10    14774392.0
11    14708279.0
12    15882180.0
13    17591559.0
14    20712135.0
15    24408574.0
16    27753672.0
17    32474078.0
18    25337064.0
19    17600756.0
20    13663050.0
21    11575727.0
22     9378082.0
23     6087134.0
Name: ridership, dtype: float64

As expected, the weekend has lower ridership.

In [27]:
(
    hourly
    .groupby("weekday")["ridership"]
    .sum()
)

weekday
Friday       53219521.0
Monday       50499123.0
Saturday     38096797.0
Sunday       30197761.0
Thursday     57819479.0
Tuesday      57316155.0
Wednesday    58017163.0
Name: ridership, dtype: float64